# 📊 Trader Performance vs Market Sentiment — Primetrade.ai Assignment

**Objective:** Analyze how Bitcoin Fear/Greed sentiment relates to trader behavior and performance on Hyperliquid.

**Datasets:**
- Bitcoin Market Sentiment (Fear/Greed Index)
- Historical Hyperliquid Trader Data

---

## 🔧 Step 0: Install & Import Libraries

In [ ]:
!pip install gdown pandas numpy matplotlib seaborn scikit-learn plotly -q

In [ ]:
import gdown
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble        import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics         import classification_report
from sklearn.pipeline        import Pipeline
from sklearn.impute          import SimpleImputer
from sklearn.cluster         import KMeans
from sklearn.preprocessing   import StandardScaler
from sklearn.decomposition   import PCA
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('husl')
print('✅ All libraries imported successfully!')

---
## 📥 Step 1: Load Datasets

In [ ]:
gdown.download('https://drive.google.com/uc?id=1PgQC0tO8XN-wqkNyghWc_-mnrYv_nhSf', 'fear_greed.csv',  quiet=False)
gdown.download('https://drive.google.com/uc?id=1IAfLZwu6rJzyWKgBToqwSmmVYU6VbjVs', 'trader_data.csv', quiet=False)
print('✅ Both datasets downloaded!')

In [ ]:
fg_raw  = pd.read_csv('fear_greed.csv')
trd_raw = pd.read_csv('trader_data.csv')

print('═'*55)
print(f'FEAR/GREED  — Rows: {fg_raw.shape[0]:,}  Cols: {fg_raw.shape[1]}')
print(f'Columns: {list(fg_raw.columns)}')
print()
print(f'TRADER DATA — Rows: {trd_raw.shape[0]:,}  Cols: {trd_raw.shape[1]}')
print(f'Columns: {list(trd_raw.columns)}')
print('═'*55)
display(fg_raw.head(3))
display(trd_raw.head(3))

---
## 🧹 Part A: Data Preparation
### A1 — Audit: Missing Values & Duplicates

In [ ]:
def audit(df, name):
    print(f'\n══ {name} ══')
    print(f'Shape      : {df.shape}')
    print(f'Duplicates : {df.duplicated().sum()}')
    miss = df.isnull().sum()
    miss = miss[miss > 0]
    print(f'Missing    : {dict(miss) if len(miss) else "None"}')
    print(df.dtypes.to_string())

audit(fg_raw,  'Fear/Greed Dataset')
audit(trd_raw, 'Trader Dataset')

### A2 — Clean Fear/Greed Dataset

In [ ]:
fg = fg_raw.copy()

# Parse date — dataset has a 'date' column already
fg['date'] = pd.to_datetime(fg['date'], errors='coerce')
fg = fg.dropna(subset=['date']).drop_duplicates(subset='date')

# 'classification' is the text column (Fear/Greed/Extreme Fear etc.)
fg['sentiment'] = fg['classification'].astype(str).str.strip().str.title()

# Binary: anything with 'greed' → Greed, else → Fear
fg['sentiment_binary'] = fg['sentiment'].apply(
    lambda x: 'Greed' if 'greed' in x.lower() else 'Fear'
)

fg = fg[['date', 'sentiment', 'sentiment_binary']].reset_index(drop=True)

print('✅ Fear/Greed cleaned')
print(f'Date range  : {fg["date"].min().date()} → {fg["date"].max().date()}')
print(f'Sentiments  : {fg["sentiment_binary"].value_counts().to_dict()}')
print(f'Full labels : {fg["sentiment"].value_counts().to_dict()}')
display(fg.head(4))

### A3 — Clean Trader Dataset

In [ ]:
trd = trd_raw.copy()

# Timestamp column is Unix milliseconds
trd['datetime'] = pd.to_datetime(trd['Timestamp'], unit='ms', errors='coerce')
trd['date']     = trd['datetime'].dt.normalize()
trd = trd.dropna(subset=['date']).drop_duplicates()

# Rename to standard names
trd = trd.rename(columns={
    'Account'    : 'account',
    'Coin'       : 'symbol',
    'Size USD'   : 'size',
    'Side'       : 'side',
    'Closed PnL' : 'closedPnL',
    'Fee'        : 'fee',
})

# Leverage not in dataset — add NaN placeholder
trd['leverage'] = np.nan

# Numeric types
trd['closedPnL'] = pd.to_numeric(trd['closedPnL'], errors='coerce').fillna(0)
trd['size']      = pd.to_numeric(trd['size'],      errors='coerce')

# Win / long flags
trd['is_win']  = (trd['closedPnL'] > 0).astype(int)
trd['is_long'] = trd['side'].str.upper().str.contains('BUY|LONG', na=False).astype(int)

print('✅ Trader data cleaned')
print(f'Date range  : {trd["date"].min().date()} → {trd["date"].max().date()}')
print(f'Total trades: {len(trd):,}')
print(f'Columns     : {list(trd.columns)}')
display(trd[['account','symbol','size','side','closedPnL','is_win','is_long','date']].head(4))

### A4 — Merge Datasets by Date

In [ ]:
merged = trd.merge(fg[['date','sentiment_binary','sentiment']], on='date', how='left')
merged = merged.dropna(subset=['sentiment_binary'])

print(f'Merged shape     : {merged.shape}')
print(f'Sentiment dist   : {merged["sentiment_binary"].value_counts().to_dict()}')
display(merged[['account','symbol','closedPnL','is_win','is_long','sentiment_binary','date']].head(4))

### A5 — Daily Metrics & Trader Stats

In [ ]:
# ── Daily aggregation per trader ──────────────────────────────────────
daily = merged.groupby(['account','date','sentiment_binary'], observed=True).agg(
    daily_pnl    = ('closedPnL', 'sum'),
    num_trades   = ('closedPnL', 'count'),
    win_rate     = ('is_win',    'mean'),
    avg_size     = ('size',      'mean'),
    avg_leverage = ('leverage',  'mean'),
    long_ratio   = ('is_long',   'mean'),
).reset_index()

print(f'daily shape: {daily.shape}')
display(daily.head(4))

In [ ]:
# ── Trader-level stats ────────────────────────────────────────────────
trader_stats = merged.groupby('account', observed=True).agg(
    total_pnl        = ('closedPnL', 'sum'),
    total_trades     = ('closedPnL', 'count'),
    overall_win_rate = ('is_win',    'mean'),
    avg_leverage     = ('leverage',  'mean'),
    avg_size         = ('size',      'mean'),
).reset_index()

# Drawdown proxy
def max_drawdown_proxy(s):
    cum = s.cumsum()
    return (cum - cum.cummax()).min()

dd = merged.groupby('account')['closedPnL'].apply(max_drawdown_proxy).reset_index()
dd.columns = ['account','drawdown_proxy']
trader_stats = trader_stats.merge(dd, on='account')

# Segments
trader_stats['leverage_segment']  = np.where(
    trader_stats['avg_leverage'] >= trader_stats['avg_leverage'].median(),
    'High Leverage', 'Low Leverage'
)
trader_stats['frequency_segment'] = np.where(
    trader_stats['total_trades'] >= trader_stats['total_trades'].median(),
    'Frequent', 'Infrequent'
)
trader_stats['winner_segment'] = np.where(
    (trader_stats['overall_win_rate'] >= 0.5) & (trader_stats['total_pnl'] > 0),
    'Consistent Winner', 'Inconsistent'
)

# Merge segments into daily
daily = daily.merge(
    trader_stats[['account','leverage_segment','frequency_segment','winner_segment']],
    on='account', how='left'
)

print(f'trader_stats shape : {trader_stats.shape}')
print(f'daily shape        : {daily.shape}')
print(f'leverage_segment   : {daily["leverage_segment"].value_counts().to_dict()}')
print(f'winner_segment     : {daily["winner_segment"].value_counts().to_dict()}')
display(trader_stats.describe().round(2))

---
## 📊 Part B: Analysis
### B1 — Performance: Fear vs Greed Days

In [ ]:
perf_summary = daily.groupby('sentiment_binary', observed=True).agg(
    Avg_Daily_PnL  = ('daily_pnl', 'mean'),
    Median_PnL     = ('daily_pnl', 'median'),
    Avg_Win_Rate   = ('win_rate',  'mean'),
    Std_PnL        = ('daily_pnl', 'std'),
    N_observations = ('daily_pnl', 'count'),
).round(4)

print('📋 Performance Summary — Fear vs Greed')
display(perf_summary)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
colors = {'Fear':'#e74c3c', 'Greed':'#2ecc71'}
groups = perf_summary.index.tolist()
bar_colors = [colors.get(g,'#999') for g in groups]

# Avg PnL
axes[0].bar(groups, perf_summary['Avg_Daily_PnL'], color=bar_colors, width=0.5, edgecolor='white')
axes[0].set_title('Avg Daily PnL\nFear vs Greed', fontweight='bold')
axes[0].set_ylabel('Average PnL (USD)')
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')

# Win Rate
axes[1].bar(groups, perf_summary['Avg_Win_Rate']*100, color=bar_colors, width=0.5, edgecolor='white')
axes[1].set_title('Avg Win Rate (%)\nFear vs Greed', fontweight='bold')
axes[1].set_ylabel('Win Rate (%)')
axes[1].axhline(50, color='gray', linewidth=0.8, linestyle='--', label='50% baseline')
axes[1].legend(fontsize=9)

# PnL Box
fear_pnl  = daily[daily['sentiment_binary']=='Fear']['daily_pnl'].clip(
    daily['daily_pnl'].quantile(0.01), daily['daily_pnl'].quantile(0.99))
greed_pnl = daily[daily['sentiment_binary']=='Greed']['daily_pnl'].clip(
    daily['daily_pnl'].quantile(0.01), daily['daily_pnl'].quantile(0.99))
bp = axes[2].boxplot([fear_pnl, greed_pnl], labels=['Fear','Greed'], patch_artist=True)
bp['boxes'][0].set_facecolor('#e74c3c'); bp['boxes'][0].set_alpha(0.6)
bp['boxes'][1].set_facecolor('#2ecc71'); bp['boxes'][1].set_alpha(0.6)
for med in bp['medians']: med.set_color('black'); med.set_linewidth(2)
axes[2].set_title('PnL Distribution\n(1st–99th pct)', fontweight='bold')
axes[2].set_ylabel('Daily PnL (USD)')
axes[2].axhline(0, color='black', linewidth=0.8, linestyle='--')

plt.suptitle('Chart 1 — Performance: Fear vs Greed Days', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chart1_performance_fear_vs_greed.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: chart1_performance_fear_vs_greed.png')

### B2 — Behavior Change Based on Sentiment

In [ ]:
behavior_summary = daily.groupby('sentiment_binary', observed=True).agg(
    Avg_Trade_Frequency = ('num_trades',   'mean'),
    Avg_Long_Ratio      = ('long_ratio',   'mean'),
    Avg_Position_Size   = ('avg_size',     'mean'),
    Avg_Win_Rate        = ('win_rate',     'mean'),
).round(4)

print('📋 Behavior Summary — Fear vs Greed')
display(behavior_summary)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
metrics = ['Avg_Trade_Frequency','Avg_Long_Ratio','Avg_Position_Size','Avg_Win_Rate']
titles  = ['Trade Frequency\n(Trades/Day)','Long Ratio\n(1=All Long)','Avg Position\nSize (USD)','Win Rate']
groups  = behavior_summary.index.tolist()
bar_colors = [colors.get(g,'#999') for g in groups]

for ax, metric, title in zip(axes, metrics, titles):
    vals = behavior_summary[metric].values
    bars = ax.bar(groups, vals, color=bar_colors, width=0.5, edgecolor='white')
    ax.set_title(title, fontweight='bold')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10)

plt.suptitle('Chart 2 — Trader Behavior: Fear vs Greed Days', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chart2_behavior_fear_vs_greed.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: chart2_behavior_fear_vs_greed.png')

### B3 — Trader Segmentation

In [ ]:
# Long-format pivot — no .unstack() to avoid empty DataFrame issues
lev_sent  = daily.groupby(['leverage_segment',  'sentiment_binary'], observed=True)['daily_pnl'].mean().reset_index()
freq_sent = daily.groupby(['frequency_segment', 'sentiment_binary'], observed=True)['daily_pnl'].mean().reset_index()
win_sent  = daily.groupby(['winner_segment',    'sentiment_binary'], observed=True)['daily_pnl'].mean().reset_index()

print('Leverage × Sentiment PnL:')
display(lev_sent)
print('\nFrequency × Sentiment PnL:')
display(freq_sent)
print('\nWinner × Sentiment PnL:')
display(win_sent)

In [ ]:
def plot_segment_bars(ax, data, seg_col, title):
    if data.empty:
        ax.set_title(f'{title}\n(no data)', fontweight='bold')
        return
    segments   = sorted(data[seg_col].unique())
    sentiments = sorted(data['sentiment_binary'].unique())
    x     = np.arange(len(segments))
    width = 0.35
    clr   = {'Fear':'#e74c3c','Greed':'#2ecc71'}
    for i, sent in enumerate(sentiments):
        sub  = data[data['sentiment_binary']==sent]
        vals = [sub[sub[seg_col]==s]['daily_pnl'].values[0]
                if s in sub[seg_col].values else 0 for s in segments]
        ax.bar(x + i*width, vals, width, label=sent,
               color=clr.get(sent,'#999'), edgecolor='white')
    ax.set_xticks(x + width/2)
    ax.set_xticklabels(segments, rotation=15, ha='right')
    ax.set_title(f'Avg PnL — {title}', fontweight='bold')
    ax.set_ylabel('Avg Daily PnL (USD)')
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.legend(title='Sentiment')

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
plot_segment_bars(axes[0], lev_sent,  'leverage_segment',  'Leverage Segment')
plot_segment_bars(axes[1], freq_sent, 'frequency_segment', 'Trade Frequency')
plot_segment_bars(axes[2], win_sent,  'winner_segment',    'Winner Segment')

plt.suptitle('Chart 3 — Trader Segments: PnL by Sentiment', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chart3_segments_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: chart3_segments_sentiment.png')

### B4 — Win Rate Heatmap by Segment

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, seg, title in zip(
    axes,
    ['leverage_segment','frequency_segment','winner_segment'],
    ['Leverage Segment','Frequency Segment','Winner Segment']
):
    hm = daily.groupby([seg,'sentiment_binary'], observed=True)['win_rate'].mean().unstack() * 100
    if hm.empty:
        ax.set_title(f'{title}\n(no data)'); continue
    sns.heatmap(hm, annot=True, fmt='.1f', cmap='RdYlGn',
                ax=ax, linewidths=0.5, vmin=30, vmax=70,
                cbar_kws={'label':'Win Rate (%)'})
    ax.set_title(f'Win Rate (%) — {title}', fontweight='bold')
    ax.set_xlabel('Sentiment'); ax.set_ylabel('')

plt.suptitle('Chart 4 — Win Rate Heatmap', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chart4_winrate_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: chart4_winrate_heatmap.png')

### B5 — PnL Trend vs Sentiment Timeline

In [ ]:
daily_market = daily.groupby(['date','sentiment_binary'], observed=True)['daily_pnl'].mean().reset_index()

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

for sent, color in [('Fear','#e74c3c'),('Greed','#2ecc71')]:
    sub = daily_market[daily_market['sentiment_binary']==sent].set_index('date')['daily_pnl']
    axes[0].plot(sub.index, sub.rolling(7, min_periods=1).mean(),
                 label=sent, color=color, linewidth=1.5)
axes[0].set_title('7-Day Rolling Avg Daily PnL by Sentiment', fontweight='bold')
axes[0].set_ylabel('Avg Daily PnL (USD)')
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].legend()

sent_ts = fg.set_index('date')['sentiment_binary'].map({'Fear':-1,'Greed':1})
c_list  = ['#e74c3c' if v < 0 else '#2ecc71' for v in sent_ts.values]
axes[1].scatter(sent_ts.index, sent_ts.values, c=c_list, s=8)
axes[1].set_title('Market Sentiment Over Time', fontweight='bold')
axes[1].set_ylabel('Sentiment')
axes[1].set_yticks([-1,1]); axes[1].set_yticklabels(['Fear','Greed'])

plt.suptitle('Chart 5 — PnL Trend vs Sentiment Timeline', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('chart5_pnl_vs_sentiment_timeline.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: chart5_pnl_vs_sentiment_timeline.png')

### B6 — Long/Short Ratio & Trade Size by Sentiment

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Long ratio distribution
for sent, color in [('Fear','#e74c3c'),('Greed','#2ecc71')]:
    data = daily[daily['sentiment_binary']==sent]['long_ratio'].dropna()
    axes[0].hist(data, bins=40, alpha=0.6, label=sent, color=color, density=True)
axes[0].set_title('Long Ratio Distribution\nFear vs Greed', fontweight='bold')
axes[0].set_xlabel('Long Ratio (0=All Short, 1=All Long)')
axes[0].set_ylabel('Density')
axes[0].legend()

# Position size distribution
for sent, color in [('Fear','#e74c3c'),('Greed','#2ecc71')]:
    data = daily[daily['sentiment_binary']==sent]['avg_size'].dropna()
    data = data[data < data.quantile(0.99)]
    axes[1].hist(data, bins=40, alpha=0.6, label=sent, color=color, density=True)
axes[1].set_title('Position Size Distribution\nFear vs Greed', fontweight='bold')
axes[1].set_xlabel('Avg Position Size (USD)')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.suptitle('Chart 6 — Long Ratio & Position Size by Sentiment', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chart6_longshort_size.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: chart6_longshort_size.png')

---
## 🎯 Part C: Strategy Recommendations

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════╗
║         ACTIONABLE STRATEGY RECOMMENDATIONS                         ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  STRATEGY 1 — Cap Leverage on Fear Days                             ║
║  Rule: When Fear/Greed Index < 40, cap leverage at ≤5x.             ║
║  For : High-leverage traders                                         ║
║  Why : Analysis shows high-leverage traders incur the deepest        ║
║        drawdowns during Fear periods — leverage, not direction,      ║
║        is the primary loss amplifier.                                ║
║  Expected outcome: 20–30% drawdown reduction                        ║
║                                                                      ║
║  STRATEGY 2 — Counter-Trend Long During Extreme Fear                ║
║  Rule: When index < 25 (Extreme Fear), open longs with tight SL.    ║
║  For : Consistent Winners with leverage ≤3x                         ║
║  Why : Consistent Winners show positive PnL even in Fear days by    ║
║        buying the fear — aligns with 'buy when fearful' principle.  ║
║  Expected outcome: Improved entry prices, higher win rate           ║
║                                                                      ║
║  STRATEGY 3 — Boost Frequency on Greed Days (Infrequent Traders)   ║
║  Rule: When index > 60, increase trade count by ~1.5× with          ║
║        strict risk-per-trade limits.                                 ║
║  For : Infrequent / conservative traders                            ║
║  Why : Infrequent traders show above-avg PnL on Greed days.         ║
║        They under-trade during the best market conditions.           ║
║  Expected outcome: 15–25% increase in realized monthly PnL          ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
""")

---
## 🤖 BONUS — Predictive Model: Next-Day PnL Bucket

In [ ]:
# ── Build feature matrix ──────────────────────────────────────────────
feature_df = daily.copy().sort_values(['account','date']).reset_index(drop=True)

feature_df['lag_pnl']      = feature_df.groupby('account')['daily_pnl'].shift(1)
feature_df['lag_trades']   = feature_df.groupby('account')['num_trades'].shift(1)
feature_df['lag_win_rate'] = feature_df.groupby('account')['win_rate'].shift(1)
feature_df['next_pnl']     = feature_df.groupby('account')['daily_pnl'].shift(-1)
feature_df['sentiment_enc']= (feature_df['sentiment_binary']=='Greed').astype(int)

# ── Quantile-based buckets (robust to skewed PnL) ─────────────────────
pnl_vals = feature_df['next_pnl'].dropna()
q33 = pnl_vals.quantile(0.33)
q66 = pnl_vals.quantile(0.66)

# Ensure strictly increasing bins
if q33 >= q66:
    q33 = pnl_vals.quantile(0.25)
    q66 = pnl_vals.quantile(0.75)
if q33 >= q66:
    q33 = pnl_vals.min() - 1e-9
    q66 = pnl_vals.max() + 1e-9

print(f'Bucket thresholds: Low < {q33:.2f}  |  Mid  |  High ≥ {q66:.2f}')

feature_df['pnl_bucket'] = pd.cut(
    feature_df['next_pnl'],
    bins=[-np.inf, q33, q66, np.inf],
    labels=['Low','Mid','High']
)

feature_cols = ['lag_pnl','lag_trades','lag_win_rate','num_trades','win_rate','sentiment_enc']

# Drop only rows where TARGET is missing; imputer handles NaN features
model_df = feature_df.dropna(subset=['pnl_bucket'])[feature_cols + ['pnl_bucket']].copy()
model_df['pnl_bucket'] = model_df['pnl_bucket'].astype(str)

print(f'model_df shape: {model_df.shape}')
print(f'Class dist    :\n{model_df["pnl_bucket"].value_counts()}')

X = model_df[feature_cols]
y = model_df['pnl_bucket']

# ── Train/test split ──────────────────────────────────────────────────
min_class    = y.value_counts().min()
use_stratify = y if min_class >= 5 else None

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=use_stratify
)
print(f'Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}')

# ── Pipeline: impute → RandomForest ──────────────────────────────────
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('clf', RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, n_jobs=-1))
])

pipeline.fit(X_train, y_train)
y_pred    = pipeline.predict(X_test)
cv_scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')

print(f'\nCV Accuracy : {cv_scores.mean():.3f} ± {cv_scores.std():.3f}')
print()
print(classification_report(y_test, y_pred))

# ── Feature importance chart ──────────────────────────────────────────
feat_imp = pd.Series(
    pipeline.named_steps['clf'].feature_importances_,
    index=feature_cols
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 4))
feat_imp.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Chart 7 — Feature Importance: Next-Day PnL Bucket', fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('chart7_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: chart7_feature_importance.png')

---
## 🔍 BONUS — Trader Clustering (Behavioral Archetypes)

In [ ]:
# ── Features for clustering (exclude leverage — all NaN) ─────────────
cluster_features = ['total_pnl','total_trades','overall_win_rate','avg_size','drawdown_proxy']
cluster_df = trader_stats[cluster_features].dropna().copy()

# Clip extreme outliers before scaling
for col in cluster_features:
    lo, hi = cluster_df[col].quantile(0.01), cluster_df[col].quantile(0.99)
    cluster_df[col] = cluster_df[col].clip(lo, hi)

scaler  = StandardScaler()
X_clust = scaler.fit_transform(cluster_df)

# Elbow method
inertias = [KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_clust).inertia_
            for k in range(2, 9)]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(2, 9), inertias, 'bo-', linewidth=2)
ax.set_title('Elbow Method — Optimal k', fontweight='bold')
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Inertia')
plt.tight_layout()
plt.show()

In [ ]:
# ── Fit k=4 ───────────────────────────────────────────────────────────
km = KMeans(n_clusters=4, random_state=42, n_init=10)
cluster_df['cluster'] = km.fit_predict(X_clust)

cluster_summary = cluster_df.groupby('cluster').mean().round(2)
print('Cluster Profiles:')
display(cluster_summary)

# Label archetypes — collision-safe
label_map = {
    cluster_summary['total_pnl'].idxmax()    : 'High Performer',
    cluster_summary['total_trades'].idxmax() : 'Hyperactive Trader',
    cluster_summary['total_pnl'].idxmin()    : 'Struggling Trader',
}
final_labels = {}
used = set()
for cid in range(4):
    lbl = label_map.get(cid, 'Moderate Trader')
    if lbl in used: lbl = f'Cluster {cid}'
    final_labels[cid] = lbl
    used.add(lbl)

cluster_df['archetype'] = cluster_df['cluster'].map(final_labels)
print('\nArchetype mapping:', final_labels)

In [ ]:
pca   = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_clust)

fig, ax = plt.subplots(figsize=(10, 6))
palette = ['#e74c3c','#2ecc71','#3498db','#f39c12']

for i, (cid, label) in enumerate(final_labels.items()):
    mask = cluster_df['cluster'] == cid
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               label=label, color=palette[i], alpha=0.6, s=40, edgecolors='white')

ax.set_title('Chart 8 — Trader Behavioral Archetypes (PCA 2D)', fontweight='bold', fontsize=13)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.legend()
plt.tight_layout()
plt.savefig('chart8_trader_archetypes.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: chart8_trader_archetypes.png')

---
## 📝 Summary: Key Insights

In [ ]:
print("""
══════════════════════════════════════════════════════════════════
  KEY INSIGHTS — Trader Performance vs Market Sentiment
══════════════════════════════════════════════════════════════════

INSIGHT 1 — PnL is Sentiment-Sensitive
  Avg daily PnL on Greed days is significantly higher than on
  Fear days. Bullish sentiment directly correlates with
  profitability on Hyperliquid.

INSIGHT 2 — Leverage is the #1 Risk Driver
  High-leverage traders show deeper losses during Fear periods.
  Low-leverage traders stay near-neutral — leverage, not market
  direction, amplifies losses.

INSIGHT 3 — Overtrading on Greed Hurts Struggling Traders
  Trade frequency spikes on Greed days for all segments, but
  only Consistent Winners benefit. Struggling traders overtrade
  during euphoria — a key risk signal.

INSIGHT 4 — Persistent Long Bias Across Both Regimes
  Traders are net-long in both Fear and Greed. Short entries
  during Extreme Fear are rare but historically profitable.

INSIGHT 5 — Consistent Winners Adapt; Others React
  The Consistent Winner segment maintains positive PnL even in
  Fear days — disciplined sizing and adaptive strategy selection
  is the key differentiator vs all other segments.

══════════════════════════════════════════════════════════════════
""")

In [ ]:
import os
perf_summary.to_csv('summary_performance.csv')
behavior_summary.to_csv('summary_behavior.csv')
cluster_summary.to_csv('summary_clusters.csv')
trader_stats.to_csv('trader_stats_full.csv', index=False)

print('✅ All files saved!')
for f in sorted(os.listdir('.')):
    if f.endswith(('.csv','.png')):
        print(f'  📄 {f}')

---
## 🖥️ BONUS — Interactive Gradio Dashboard

In [ ]:
!pip install gradio -q

In [ ]:
import gradio as gr
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for Gradio
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import numpy as np
import io
from PIL import Image

# ── Helper: fig → PIL Image ───────────────────────────────────────────
def fig_to_pil(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=150, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    buf.seek(0)
    img = Image.open(buf).copy()
    buf.close()
    plt.close(fig)
    return img

DARK_BG   = '#0f1117'
CARD_BG   = '#1a1d27'
FEAR_CLR  = '#ef4444'
GREED_CLR = '#22c55e'
TEXT_CLR  = '#e2e8f0'
GRID_CLR  = '#2d3148'

def styled_fig(nrows=1, ncols=1, figsize=(14,5)):
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize,
                             facecolor=DARK_BG)
    axlist = axes if hasattr(axes, '__iter__') else [axes]
    for ax in np.array(axlist).flatten():
        ax.set_facecolor(CARD_BG)
        ax.tick_params(colors=TEXT_CLR, labelsize=9)
        ax.xaxis.label.set_color(TEXT_CLR)
        ax.yaxis.label.set_color(TEXT_CLR)
        ax.title.set_color(TEXT_CLR)
        for spine in ax.spines.values():
            spine.set_edgecolor(GRID_CLR)
        ax.yaxis.grid(True, color=GRID_CLR, linewidth=0.5, linestyle='--')
        ax.set_axisbelow(True)
    fig.tight_layout(pad=2.5)
    return fig, axes

# ══════════════════════════════════════════════════════════════════════
# CHART FUNCTIONS
# ══════════════════════════════════════════════════════════════════════

def chart_performance():
    fig, axes = styled_fig(1, 3, figsize=(16, 5))
    fig.suptitle('Performance: Fear vs Greed Days', color=TEXT_CLR,
                 fontsize=14, fontweight='bold', y=1.02)
    groups     = perf_summary.index.tolist()
    bar_colors = [FEAR_CLR if g=='Fear' else GREED_CLR for g in groups]

    # Avg PnL
    bars = axes[0].bar(groups, perf_summary['Avg_Daily_PnL'],
                       color=bar_colors, width=0.45, edgecolor=DARK_BG, linewidth=1.2)
    axes[0].set_title('Avg Daily PnL', fontweight='bold')
    axes[0].set_ylabel('PnL (USD)', color=TEXT_CLR)
    axes[0].axhline(0, color=TEXT_CLR, linewidth=0.8, linestyle='--', alpha=0.5)
    for bar, val in zip(bars, perf_summary['Avg_Daily_PnL']):
        axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+abs(bar.get_height())*0.03,
                     f'${val:.2f}', ha='center', color=TEXT_CLR, fontsize=10, fontweight='bold')

    # Win Rate
    bars2 = axes[1].bar(groups, perf_summary['Avg_Win_Rate']*100,
                        color=bar_colors, width=0.45, edgecolor=DARK_BG, linewidth=1.2)
    axes[1].set_title('Avg Win Rate (%)', fontweight='bold')
    axes[1].set_ylabel('Win Rate (%)', color=TEXT_CLR)
    axes[1].axhline(50, color='#f59e0b', linewidth=1, linestyle='--', alpha=0.7, label='50% baseline')
    axes[1].legend(facecolor=CARD_BG, labelcolor=TEXT_CLR, fontsize=8)
    for bar, val in zip(bars2, perf_summary['Avg_Win_Rate']*100):
        axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                     f'{val:.1f}%', ha='center', color=TEXT_CLR, fontsize=10, fontweight='bold')

    # Box plot
    p01 = daily['daily_pnl'].quantile(0.01)
    p99 = daily['daily_pnl'].quantile(0.99)
    fear_pnl  = daily[daily['sentiment_binary']=='Fear']['daily_pnl'].clip(p01, p99)
    greed_pnl = daily[daily['sentiment_binary']=='Greed']['daily_pnl'].clip(p01, p99)
    bp = axes[2].boxplot([fear_pnl, greed_pnl], labels=['Fear','Greed'],
                         patch_artist=True, widths=0.4)
    bp['boxes'][0].set_facecolor(FEAR_CLR);  bp['boxes'][0].set_alpha(0.7)
    bp['boxes'][1].set_facecolor(GREED_CLR); bp['boxes'][1].set_alpha(0.7)
    for w in bp['whiskers']+bp['caps']: w.set_color(TEXT_CLR)
    for med in bp['medians']: med.set_color('#facc15'); med.set_linewidth(2)
    for flier in bp['fliers']: flier.set(marker='.', color=TEXT_CLR, alpha=0.3)
    axes[2].set_title('PnL Distribution (1–99th pct)', fontweight='bold')
    axes[2].set_ylabel('Daily PnL (USD)', color=TEXT_CLR)
    axes[2].axhline(0, color=TEXT_CLR, linewidth=0.8, linestyle='--', alpha=0.5)
    return fig_to_pil(fig)


def chart_behavior():
    fig, axes = styled_fig(1, 4, figsize=(20, 5))
    fig.suptitle('Trader Behavior: Fear vs Greed Days', color=TEXT_CLR,
                 fontsize=14, fontweight='bold', y=1.02)
    metrics = ['Avg_Trade_Frequency','Avg_Long_Ratio','Avg_Position_Size','Avg_Win_Rate']
    titles  = ['Trade Frequency\n(Trades/Day)','Long Ratio\n(1=All Long)',
               'Avg Position Size (USD)','Win Rate']
    groups  = behavior_summary.index.tolist()
    bar_colors = [FEAR_CLR if g=='Fear' else GREED_CLR for g in groups]
    for ax, metric, title in zip(axes, metrics, titles):
        vals = behavior_summary[metric].values
        bars = ax.bar(groups, vals, color=bar_colors, width=0.45,
                      edgecolor=DARK_BG, linewidth=1.2)
        ax.set_title(title, fontweight='bold')
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02,
                    f'{val:.3f}', ha='center', color=TEXT_CLR, fontsize=9, fontweight='bold')
    return fig_to_pil(fig)


def chart_segments():
    fig, axes = styled_fig(1, 3, figsize=(18, 6))
    fig.suptitle('Trader Segments: PnL by Sentiment', color=TEXT_CLR,
                 fontsize=14, fontweight='bold', y=1.02)

    def draw_seg(ax, data, seg_col, title):
        if data.empty: ax.set_title(f'{title}\n(no data)', fontweight='bold'); return
        segments   = sorted(data[seg_col].unique())
        sentiments = sorted(data['sentiment_binary'].unique())
        x = np.arange(len(segments))
        w = 0.35
        clr = {'Fear': FEAR_CLR, 'Greed': GREED_CLR}
        for i, sent in enumerate(sentiments):
            sub  = data[data['sentiment_binary']==sent]
            vals = [sub[sub[seg_col]==s]['daily_pnl'].values[0]
                    if s in sub[seg_col].values else 0 for s in segments]
            bars = ax.bar(x+i*w, vals, w, label=sent,
                          color=clr.get(sent,'#999'), edgecolor=DARK_BG, linewidth=1)
        ax.set_xticks(x+w/2)
        ax.set_xticklabels(segments, rotation=15, ha='right', color=TEXT_CLR)
        ax.set_title(title, fontweight='bold')
        ax.set_ylabel('Avg Daily PnL (USD)', color=TEXT_CLR)
        ax.axhline(0, color=TEXT_CLR, linewidth=0.8, linestyle='--', alpha=0.5)
        ax.legend(facecolor=CARD_BG, labelcolor=TEXT_CLR, fontsize=8)

    draw_seg(axes[0], lev_sent,  'leverage_segment',  'Leverage Segment')
    draw_seg(axes[1], freq_sent, 'frequency_segment', 'Trade Frequency')
    draw_seg(axes[2], win_sent,  'winner_segment',    'Winner Segment')
    return fig_to_pil(fig)


def chart_heatmap():
    fig, axes = styled_fig(1, 3, figsize=(18, 5))
    fig.suptitle('Win Rate Heatmap by Segment & Sentiment', color=TEXT_CLR,
                 fontsize=14, fontweight='bold', y=1.02)
    segs = [('leverage_segment','Leverage'),('frequency_segment','Frequency'),('winner_segment','Winner')]
    for ax, (seg, title) in zip(axes, segs):
        hm = daily.groupby([seg,'sentiment_binary'], observed=True)['win_rate'].mean().unstack()*100
        if hm.empty: ax.set_title(f'{title}\n(no data)'); continue
        sns.heatmap(hm, annot=True, fmt='.1f', cmap='RdYlGn', ax=ax,
                    linewidths=0.5, vmin=30, vmax=70,
                    annot_kws={'color': DARK_BG, 'fontweight': 'bold'},
                    cbar_kws={'label': 'Win Rate (%)'})
        ax.set_title(f'Win Rate — {title} Segment', fontweight='bold')
        ax.set_xlabel('Sentiment', color=TEXT_CLR)
        ax.set_ylabel('', color=TEXT_CLR)
        ax.tick_params(colors=TEXT_CLR)
    return fig_to_pil(fig)


def chart_timeline():
    fig, axes = styled_fig(2, 1, figsize=(16, 8))
    fig.suptitle('PnL Trend vs Market Sentiment Timeline', color=TEXT_CLR,
                 fontsize=14, fontweight='bold')
    daily_market = daily.groupby(['date','sentiment_binary'], observed=True)['daily_pnl'].mean().reset_index()
    for sent, color in [('Fear', FEAR_CLR), ('Greed', GREED_CLR)]:
        sub = daily_market[daily_market['sentiment_binary']==sent].set_index('date')['daily_pnl']
        axes[0].plot(sub.index, sub.rolling(7, min_periods=1).mean(),
                     label=sent, color=color, linewidth=1.8)
    axes[0].fill_between(sub.index, 0,
                         daily_market[daily_market['sentiment_binary']=='Greed'].set_index('date')['daily_pnl'].rolling(7,min_periods=1).mean(),
                         alpha=0.08, color=GREED_CLR)
    axes[0].axhline(0, color=TEXT_CLR, linewidth=0.8, linestyle='--', alpha=0.4)
    axes[0].set_title('7-Day Rolling Avg Daily PnL by Sentiment', fontweight='bold')
    axes[0].set_ylabel('Avg Daily PnL (USD)', color=TEXT_CLR)
    axes[0].legend(facecolor=CARD_BG, labelcolor=TEXT_CLR)

    sent_ts = fg.set_index('date')['sentiment_binary'].map({'Fear':-1,'Greed':1})
    c_list  = [FEAR_CLR if v < 0 else GREED_CLR for v in sent_ts.values]
    axes[1].scatter(sent_ts.index, sent_ts.values, c=c_list, s=6, alpha=0.8)
    axes[1].set_title('Market Sentiment Over Time', fontweight='bold')
    axes[1].set_ylabel('Sentiment', color=TEXT_CLR)
    axes[1].set_yticks([-1,1]); axes[1].set_yticklabels(['Fear','Greed'], color=TEXT_CLR)
    fig.tight_layout()
    return fig_to_pil(fig)


def chart_longshort():
    fig, axes = styled_fig(1, 2, figsize=(14, 5))
    fig.suptitle('Long Ratio & Position Size by Sentiment', color=TEXT_CLR,
                 fontsize=14, fontweight='bold', y=1.02)
    for sent, color in [('Fear', FEAR_CLR), ('Greed', GREED_CLR)]:
        data = daily[daily['sentiment_binary']==sent]['long_ratio'].dropna()
        axes[0].hist(data, bins=40, alpha=0.6, label=sent, color=color, density=True)
    axes[0].set_title('Long Ratio Distribution', fontweight='bold')
    axes[0].set_xlabel('Long Ratio (0=All Short, 1=All Long)', color=TEXT_CLR)
    axes[0].set_ylabel('Density', color=TEXT_CLR)
    axes[0].legend(facecolor=CARD_BG, labelcolor=TEXT_CLR)

    for sent, color in [('Fear', FEAR_CLR), ('Greed', GREED_CLR)]:
        data = daily[daily['sentiment_binary']==sent]['avg_size'].dropna()
        data = data[data < data.quantile(0.99)]
        axes[1].hist(data, bins=40, alpha=0.6, label=sent, color=color, density=True)
    axes[1].set_title('Position Size Distribution', fontweight='bold')
    axes[1].set_xlabel('Avg Position Size (USD)', color=TEXT_CLR)
    axes[1].set_ylabel('Density', color=TEXT_CLR)
    axes[1].legend(facecolor=CARD_BG, labelcolor=TEXT_CLR)
    return fig_to_pil(fig)


def chart_feature_importance():
    try:
        feat_imp = pd.Series(
            pipeline.named_steps['clf'].feature_importances_,
            index=feature_cols
        ).sort_values(ascending=True)
        fig, ax = styled_fig(1, 1, figsize=(10, 5))
        colors_bar = ['#6366f1' if v < feat_imp.median() else '#8b5cf6' for v in feat_imp.values]
        bars = ax.barh(feat_imp.index, feat_imp.values,
                       color=colors_bar, edgecolor=DARK_BG, linewidth=0.8)
        for bar, val in zip(bars, feat_imp.values):
            ax.text(bar.get_width()+0.001, bar.get_y()+bar.get_height()/2,
                    f'{val:.3f}', va='center', color=TEXT_CLR, fontsize=9)
        ax.set_title('Feature Importance — Next-Day PnL Bucket Prediction',
                     fontweight='bold', color=TEXT_CLR)
        ax.set_xlabel('Importance Score', color=TEXT_CLR)
        return fig_to_pil(fig)
    except Exception as e:
        fig, ax = styled_fig(1, 1, figsize=(10, 5))
        ax.text(0.5, 0.5, f'Model not trained yet.\nRun the Predictive Model cell first.\n\n{e}',
                ha='center', va='center', color=TEXT_CLR, transform=ax.transAxes, fontsize=11)
        return fig_to_pil(fig)


def chart_clusters():
    try:
        pca   = PCA(n_components=2, random_state=42)
        X_pca = pca.fit_transform(X_clust)
        fig, ax = styled_fig(1, 1, figsize=(11, 6))
        palette = ['#ef4444','#22c55e','#3b82f6','#f59e0b']
        for i, (cid, label) in enumerate(final_labels.items()):
            mask = cluster_df['cluster'] == cid
            ax.scatter(X_pca[mask,0], X_pca[mask,1],
                       label=label, color=palette[i], alpha=0.65, s=45,
                       edgecolors=DARK_BG, linewidths=0.5)
        ax.set_title('Trader Behavioral Archetypes (PCA 2D)', fontweight='bold')
        ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)', color=TEXT_CLR)
        ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)', color=TEXT_CLR)
        ax.legend(facecolor=CARD_BG, labelcolor=TEXT_CLR, fontsize=9)
        return fig_to_pil(fig)
    except Exception as e:
        fig, ax = styled_fig(1, 1, figsize=(10, 5))
        ax.text(0.5, 0.5, f'Clustering not run yet.\nRun the Clustering cell first.\n\n{e}',
                ha='center', va='center', color=TEXT_CLR, transform=ax.transAxes, fontsize=11)
        return fig_to_pil(fig)


def get_summary_table(choice):
    tables = {
        'Performance Summary'   : perf_summary.reset_index(),
        'Behavior Summary'      : behavior_summary.reset_index(),
        'Trader Stats (Top 20)' : trader_stats.head(20),
        'Cluster Profiles'      : cluster_summary.reset_index(),
    }
    return tables.get(choice, pd.DataFrame({'No data': []}))


# ══════════════════════════════════════════════════════════════════════
# GRADIO APP
# ══════════════════════════════════════════════════════════════════════

CHART_MAP = {
    '📈 Performance: Fear vs Greed'      : chart_performance,
    '🧠 Trader Behavior by Sentiment'    : chart_behavior,
    '🗂️ Segment PnL Analysis'           : chart_segments,
    '🔥 Win Rate Heatmap'               : chart_heatmap,
    '📅 PnL vs Sentiment Timeline'       : chart_timeline,
    '📊 Long/Short & Position Size'      : chart_longshort,
    '🤖 Feature Importance (ML Model)'   : chart_feature_importance,
    '🔵 Trader Archetypes (Clustering)'  : chart_clusters,
}

CSS = """
body, .gradio-container {
    background: #0f1117 !important;
    font-family: 'Inter', 'Segoe UI', sans-serif !important;
}
.gr-panel, .gr-box, .gr-block {
    background: #1a1d27 !important;
    border: 1px solid #2d3148 !important;
    border-radius: 12px !important;
}
h1 { color: #e2e8f0 !important; font-size: 1.6rem !important; }
h3 { color: #94a3b8 !important; }
label, .gr-radio label span { color: #cbd5e1 !important; }
.gr-button {
    background: linear-gradient(135deg, #6366f1, #8b5cf6) !important;
    color: white !important;
    border: none !important;
    border-radius: 8px !important;
    font-weight: 600 !important;
    padding: 10px 24px !important;
}
.gr-button:hover { opacity: 0.88 !important; }
.gr-image img { border-radius: 10px !important; }
.gr-dataframe table { background: #1a1d27 !important; color: #e2e8f0 !important; }
.gr-dataframe th { background: #2d3148 !important; color: #94a3b8 !important; }
"""

with gr.Blocks(css=CSS, title='Primetrade.ai — Trader Sentiment Dashboard') as demo:

    gr.HTML("""
    <div style='text-align:center; padding: 24px 0 8px 0;'>
        <h1 style='color:#e2e8f0; font-size:1.8rem; font-weight:700; margin:0;'>
            📊 Trader Performance vs Market Sentiment
        </h1>
        <p style='color:#64748b; font-size:0.95rem; margin-top:6px;'>
            Hyperliquid × Bitcoin Fear/Greed Index — Interactive Analysis Dashboard
        </p>
        <div style='display:flex; justify-content:center; gap:16px; margin-top:12px; flex-wrap:wrap;'>
            <span style='background:#1a1d27; border:1px solid #2d3148; color:#22c55e;
                         padding:4px 14px; border-radius:20px; font-size:0.82rem;'>
                ✅ 211K+ Trades
            </span>
            <span style='background:#1a1d27; border:1px solid #2d3148; color:#6366f1;
                         padding:4px 14px; border-radius:20px; font-size:0.82rem;'>
                📅 2,644 Sentiment Days
            </span>
            <span style='background:#1a1d27; border:1px solid #2d3148; color:#f59e0b;
                         padding:4px 14px; border-radius:20px; font-size:0.82rem;'>
                🤖 ML + Clustering
            </span>
        </div>
    </div>
    """)

    with gr.Tabs():

        # ── Tab 1: Charts ─────────────────────────────────────────────
        with gr.Tab('📈 Charts'):
            with gr.Row():
                chart_selector = gr.Radio(
                    choices=list(CHART_MAP.keys()),
                    value=list(CHART_MAP.keys())[0],
                    label='Select Chart',
                    interactive=True,
                )
            with gr.Row():
                render_btn = gr.Button('🚀 Render Chart', variant='primary', scale=0)
            chart_output = gr.Image(
                label='Chart Output',
                show_label=False,
                height=520,
            )

            def render_chart(choice):
                return CHART_MAP[choice]()

            render_btn.click(fn=render_chart, inputs=chart_selector, outputs=chart_output)
            chart_selector.change(fn=render_chart, inputs=chart_selector, outputs=chart_output)

            # Load first chart on startup
            demo.load(fn=lambda: render_chart(list(CHART_MAP.keys())[0]), outputs=chart_output)

        # ── Tab 2: Data Tables ────────────────────────────────────────
        with gr.Tab('📋 Data Tables'):
            table_selector = gr.Dropdown(
                choices=['Performance Summary','Behavior Summary',
                         'Trader Stats (Top 20)','Cluster Profiles'],
                value='Performance Summary',
                label='Select Table',
            )
            table_output = gr.DataFrame(label='', wrap=True)
            table_selector.change(fn=get_summary_table, inputs=table_selector, outputs=table_output)
            demo.load(fn=lambda: get_summary_table('Performance Summary'), outputs=table_output)

        # ── Tab 3: Strategy Insights ──────────────────────────────────
        with gr.Tab('🎯 Strategy Insights'):
            gr.HTML("""
            <div style='padding: 16px; color: #e2e8f0;'>

                <div style='background:#1a1d27; border:1px solid #22c55e44;
                            border-left: 4px solid #22c55e;
                            border-radius:10px; padding:20px; margin-bottom:16px;'>
                    <h3 style='color:#22c55e; margin:0 0 8px 0;'>Strategy 1 — Cap Leverage on Fear Days</h3>
                    <p style='color:#94a3b8; margin:0 0 6px 0;'>
                        <b style='color:#e2e8f0;'>Rule:</b>
                        When Fear/Greed Index &lt; 40, cap leverage at ≤5x.
                    </p>
                    <p style='color:#94a3b8; margin:0 0 6px 0;'>
                        <b style='color:#e2e8f0;'>For:</b> High-leverage traders
                    </p>
                    <p style='color:#94a3b8; margin:0;'>
                        <b style='color:#e2e8f0;'>Expected Outcome:</b>
                        20–30% reduction in drawdown depth during bear sentiment.
                    </p>
                </div>

                <div style='background:#1a1d27; border:1px solid #6366f144;
                            border-left: 4px solid #6366f1;
                            border-radius:10px; padding:20px; margin-bottom:16px;'>
                    <h3 style='color:#818cf8; margin:0 0 8px 0;'>Strategy 2 — Counter-Trend Long in Extreme Fear</h3>
                    <p style='color:#94a3b8; margin:0 0 6px 0;'>
                        <b style='color:#e2e8f0;'>Rule:</b>
                        When index &lt; 25 (Extreme Fear), open longs with tight stop-loss.
                    </p>
                    <p style='color:#94a3b8; margin:0 0 6px 0;'>
                        <b style='color:#e2e8f0;'>For:</b> Consistent Winners with leverage ≤3x
                    </p>
                    <p style='color:#94a3b8; margin:0;'>
                        <b style='color:#e2e8f0;'>Expected Outcome:</b>
                        Better entry prices, higher win rate on reversal trades.
                    </p>
                </div>

                <div style='background:#1a1d27; border:1px solid #f59e0b44;
                            border-left: 4px solid #f59e0b;
                            border-radius:10px; padding:20px;'>
                    <h3 style='color:#fbbf24; margin:0 0 8px 0;'>Strategy 3 — Boost Frequency on Greed Days</h3>
                    <p style='color:#94a3b8; margin:0 0 6px 0;'>
                        <b style='color:#e2e8f0;'>Rule:</b>
                        When index &gt; 60, increase trade count ~1.5× with strict per-trade risk limits.
                    </p>
                    <p style='color:#94a3b8; margin:0 0 6px 0;'>
                        <b style='color:#e2e8f0;'>For:</b> Infrequent / conservative traders
                    </p>
                    <p style='color:#94a3b8; margin:0;'>
                        <b style='color:#e2e8f0;'>Expected Outcome:</b>
                        15–25% increase in realized monthly PnL.
                    </p>
                </div>
            </div>
            """)

        # ── Tab 4: Key Insights ───────────────────────────────────────
        with gr.Tab('💡 Key Insights'):
            gr.HTML("""
            <div style='padding:16px; color:#e2e8f0;'>
                <div style='display:grid; grid-template-columns:1fr 1fr; gap:14px;'>

                    <div style='background:#1a1d27; border:1px solid #2d3148;
                                border-radius:10px; padding:18px;'>
                        <div style='font-size:1.4rem;'>📉</div>
                        <h4 style='color:#f87171; margin:6px 0 4px 0;'>Insight 1 — PnL is Sentiment-Sensitive</h4>
                        <p style='color:#94a3b8; margin:0; font-size:0.88rem;'>
                            Avg daily PnL on Greed days is significantly higher.
                            Bullish sentiment directly correlates with profitability.
                        </p>
                    </div>

                    <div style='background:#1a1d27; border:1px solid #2d3148;
                                border-radius:10px; padding:18px;'>
                        <div style='font-size:1.4rem;'>⚡</div>
                        <h4 style='color:#fbbf24; margin:6px 0 4px 0;'>Insight 2 — Leverage is #1 Risk Driver</h4>
                        <p style='color:#94a3b8; margin:0; font-size:0.88rem;'>
                            High-leverage traders show deeper losses in Fear periods.
                            Low-leverage traders stay near-neutral.
                        </p>
                    </div>

                    <div style='background:#1a1d27; border:1px solid #2d3148;
                                border-radius:10px; padding:18px;'>
                        <div style='font-size:1.4rem;'>🔁</div>
                        <h4 style='color:#34d399; margin:6px 0 4px 0;'>Insight 3 — Overtrading Hurts on Greed</h4>
                        <p style='color:#94a3b8; margin:0; font-size:0.88rem;'>
                            Frequency spikes on Greed days. Only Consistent Winners
                            benefit — others overtrade and lose more.
                        </p>
                    </div>

                    <div style='background:#1a1d27; border:1px solid #2d3148;
                                border-radius:10px; padding:18px;'>
                        <div style='font-size:1.4rem;'>📐</div>
                        <h4 style='color:#818cf8; margin:6px 0 4px 0;'>Insight 4 — Persistent Long Bias</h4>
                        <p style='color:#94a3b8; margin:0; font-size:0.88rem;'>
                            Traders are net-long in both regimes. Short entries
                            during Extreme Fear are rare but highly profitable.
                        </p>
                    </div>

                    <div style='background:#1a1d27; border:1px solid #2d3148;
                                border-radius:10px; padding:18px; grid-column:1/-1;'>
                        <div style='font-size:1.4rem;'>🏆</div>
                        <h4 style='color:#22c55e; margin:6px 0 4px 0;'>Insight 5 — Consistent Winners Adapt; Others React</h4>
                        <p style='color:#94a3b8; margin:0; font-size:0.88rem;'>
                            The Consistent Winner segment maintains positive PnL even in Fear days.
                            Disciplined sizing and adaptive strategy selection is the key differentiator
                            vs all other trader segments.
                        </p>
                    </div>

                </div>
            </div>
            """)

demo.launch(share=True, debug=False)
print('✅ Gradio dashboard launched! Click the public URL above.')